# Sinapsa — Brussels ↔ Chișinău weather bridge

## New geographic scope

We no longer fetch every Belgian commune.

### Belgium
We use **11 geographic areas**:
- Brussels-Capital Region
- 10 provinces

For weather, each area is represented by a reference city:
Brussels, Antwerp, Hasselt, Ghent, Bruges, Leuven, Mons, Liège, Arlon, Namur and Wavre.

### Moldova
We use:
- 32 raion centres
- Chișinău municipality
- Bălți municipality
- Comrat for Gagauzia

That gives **35 Moldovan reference centres**.

### Total
- Belgium: 11
- Moldova: 35
- Total: 46 locations
- 7-day forecast: 46 × 7 = **322 forecast rows**

The default public comparison remains:

**Brussels ↔ Chișinău**


## 1. Imports and API addresses

We now use only:
- `requests` for API calls;
- `pandas` for the table;
- `matplotlib` / `seaborn` for charts;
- Open-Meteo Geocoding API for coordinates;
- Open-Meteo Forecast API for weather.

The ODWB commune API is no longer needed for this version.


In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"


## 2. Define the 46 reference areas

The `area` is what we want to show to the user.

The `representative_city` is the city whose coordinates will be used for the weather forecast.

This distinction is important:

> We are showing **weather at a representative regional centre**, not an average weather value for the whole province or raion.


In [ ]:
reference_areas = [
    # --------------------------------------------------
    # BELGIUM — Brussels-Capital + 10 provinces
    # --------------------------------------------------
    {
        "country": "Belgium",
        "country_code": "BE",
        "area_type": "Region",
        "area": "Brussels-Capital Region",
        "representative_city": "Brussels",
    },
    {
        "country": "Belgium",
        "country_code": "BE",
        "area_type": "Province",
        "area": "Antwerp Province",
        "representative_city": "Antwerp",
    },
    {
        "country": "Belgium",
        "country_code": "BE",
        "area_type": "Province",
        "area": "Limburg Province",
        "representative_city": "Hasselt",
    },
    {
        "country": "Belgium",
        "country_code": "BE",
        "area_type": "Province",
        "area": "East Flanders Province",
        "representative_city": "Ghent",
    },
    {
        "country": "Belgium",
        "country_code": "BE",
        "area_type": "Province",
        "area": "West Flanders Province",
        "representative_city": "Bruges",
    },
    {
        "country": "Belgium",
        "country_code": "BE",
        "area_type": "Province",
        "area": "Flemish Brabant Province",
        "representative_city": "Leuven",
    },
    {
        "country": "Belgium",
        "country_code": "BE",
        "area_type": "Province",
        "area": "Hainaut Province",
        "representative_city": "Mons",
    },
    {
        "country": "Belgium",
        "country_code": "BE",
        "area_type": "Province",
        "area": "Liège Province",
        "representative_city": "Liège",
    },
    {
        "country": "Belgium",
        "country_code": "BE",
        "area_type": "Province",
        "area": "Luxembourg Province",
        "representative_city": "Arlon",
    },
    {
        "country": "Belgium",
        "country_code": "BE",
        "area_type": "Province",
        "area": "Namur Province",
        "representative_city": "Namur",
    },
    {
        "country": "Belgium",
        "country_code": "BE",
        "area_type": "Province",
        "area": "Walloon Brabant Province",
        "representative_city": "Wavre",
    },

    # --------------------------------------------------
    # MOLDOVA — municipalities / Gagauzia / 32 raion centres
    # --------------------------------------------------
    {
        "country": "Moldova",
        "country_code": "MD",
        "area_type": "Municipality",
        "area": "Chișinău Municipality",
        "representative_city": "Chișinău",
    },
    {
        "country": "Moldova",
        "country_code": "MD",
        "area_type": "Municipality",
        "area": "Bălți Municipality",
        "representative_city": "Bălți",
    },
    {
        "country": "Moldova",
        "country_code": "MD",
        "area_type": "Autonomous territorial unit",
        "area": "Gagauzia",
        "representative_city": "Comrat",
    },

    # 32 raion centres
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Anenii Noi Raion", "representative_city": "Anenii Noi"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Basarabeasca Raion", "representative_city": "Basarabeasca"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Briceni Raion", "representative_city": "Briceni"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Cahul Raion", "representative_city": "Cahul"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Călărași Raion", "representative_city": "Călărași"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Cantemir Raion", "representative_city": "Cantemir"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Căușeni Raion", "representative_city": "Căușeni"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Cimișlia Raion", "representative_city": "Cimișlia"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Criuleni Raion", "representative_city": "Criuleni"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Dondușeni Raion", "representative_city": "Dondușeni"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Drochia Raion", "representative_city": "Drochia"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Dubăsari Raion", "representative_city": "Cocieri"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Edineț Raion", "representative_city": "Edineț"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Fălești Raion", "representative_city": "Fălești"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Florești Raion", "representative_city": "Florești"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Glodeni Raion", "representative_city": "Glodeni"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Hîncești Raion", "representative_city": "Hîncești"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Ialoveni Raion", "representative_city": "Ialoveni"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Leova Raion", "representative_city": "Leova"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Nisporeni Raion", "representative_city": "Nisporeni"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Ocnița Raion", "representative_city": "Ocnița"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Orhei Raion", "representative_city": "Orhei"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Rezina Raion", "representative_city": "Rezina"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Rîșcani Raion", "representative_city": "Rîșcani"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Sîngerei Raion", "representative_city": "Sîngerei"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Soroca Raion", "representative_city": "Soroca"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Strășeni Raion", "representative_city": "Strășeni"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Șoldănești Raion", "representative_city": "Șoldănești"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Ștefan Vodă Raion", "representative_city": "Ștefan Vodă"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Taraclia Raion", "representative_city": "Taraclia"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Telenești Raion", "representative_city": "Telenești"},
    {"country": "Moldova", "country_code": "MD", "area_type": "Raion", "area": "Ungheni Raion", "representative_city": "Ungheni"},
]

print("Reference areas:", len(reference_areas))
print("Expected:", 46)


## 3. Geocode one representative city

Instead of storing coordinates manually, we ask the Open-Meteo Geocoding API.

The `countryCode` parameter prevents confusion between cities with similar names in different countries.


In [ ]:
def geocode_location(city, country_code):
    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json",
        "countryCode": country_code,
    }

    response = requests.get(
        GEOCODING_URL,
        params=params,
        timeout=20,
    )

    response.raise_for_status()
    data = response.json()

    if "results" not in data or len(data["results"]) == 0:
        raise ValueError(
            f"No geocoding result found for {city} ({country_code})"
        )

    result = data["results"][0]

    return {
        "latitude": result["latitude"],
        "longitude": result["longitude"],
        "timezone": result.get("timezone"),
        "geocoded_name": result.get("name"),
    }


## 4. Test geocoding with the default pair

Before geocoding all 46 centres, test only:

- Brussels
- Chișinău


In [ ]:
brussels_geo = geocode_location("Brussels", "BE")
chisinau_geo = geocode_location("Chișinău", "MD")

print("Brussels:", brussels_geo)
print("Chișinău:", chisinau_geo)


## 5. Geocode all 46 reference centres

This creates a clean location table.

If one place cannot be found, the `try/except` block records the error instead of stopping the whole notebook.


In [ ]:
location_records = []

for area in reference_areas:
    try:
        geo = geocode_location(
            area["representative_city"],
            area["country_code"],
        )

        record = {
            **area,
            "latitude": geo["latitude"],
            "longitude": geo["longitude"],
            "timezone": geo["timezone"],
            "geocoded_name": geo["geocoded_name"],
            "geocoding_status": "OK",
        }

    except Exception as error:
        record = {
            **area,
            "latitude": None,
            "longitude": None,
            "timezone": None,
            "geocoded_name": None,
            "geocoding_status": str(error),
        }

    location_records.append(record)

df_locations = pd.DataFrame(location_records)

print("Locations:", len(df_locations))
print("Successfully geocoded:", df_locations["latitude"].notna().sum())

df_locations


## 6. Check for geocoding problems

The next cell shows only rows where coordinates were not found.

If the result is an empty DataFrame, all 46 locations were successfully geocoded.


In [ ]:
geocoding_errors = df_locations[
    df_locations["latitude"].isna()
]

geocoding_errors


## 7. Weather-code descriptions

Open-Meteo returns numerical WMO weather codes.

We translate them into readable text for the chart and later for the website.


In [ ]:
weather_descriptions = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    56: "Light freezing drizzle",
    57: "Dense freezing drizzle",
    61: "Light rain",
    63: "Moderate rain",
    65: "Heavy rain",
    66: "Light freezing rain",
    67: "Heavy freezing rain",
    71: "Light snow",
    73: "Moderate snow",
    75: "Heavy snow",
    77: "Snow grains",
    80: "Light rain showers",
    81: "Moderate rain showers",
    82: "Heavy rain showers",
    85: "Light snow showers",
    86: "Heavy snow showers",
    95: "Thunderstorm",
    96: "Thunderstorm with light hail",
    99: "Thunderstorm with heavy hail",
}


## 8. Function: get a 7-day forecast

Input:
- one row from `df_locations`

Output:
- 7 dictionaries, one per forecast day.


In [ ]:
def get_weather(location):
    weather_params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "daily": (
            "temperature_2m_max,"
            "temperature_2m_min,"
            "precipitation_sum,"
            "precipitation_probability_max,"
            "weather_code"
        ),
        "timezone": "auto",
        "forecast_days": 7,
    }

    response = requests.get(
        FORECAST_URL,
        params=weather_params,
        timeout=20,
    )

    response.raise_for_status()
    weather_data = response.json()
    daily = weather_data["daily"]

    records = []

    for i in range(len(daily["time"])):
        records.append({
            "country": location["country"],
            "country_code": location["country_code"],
            "area_type": location["area_type"],
            "area": location["area"],
            "representative_city": location["representative_city"],
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "date": daily["time"][i],
            "temp_min": daily["temperature_2m_min"][i],
            "temp_max": daily["temperature_2m_max"][i],
            "precipitation_mm": daily["precipitation_sum"][i],
            "precipitation_probability": daily[
                "precipitation_probability_max"
            ][i],
            "weather_code": daily["weather_code"][i],
        })

    return records


## 9. Test weather with Brussels only

We still test the function with one location before processing all 46.


In [ ]:
brussels_location = df_locations[
    df_locations["area"] == "Brussels-Capital Region"
].iloc[0]

brussels_forecast = get_weather(brussels_location)

print("Forecast rows:", len(brussels_forecast))
print(brussels_forecast[0])


## 10. Get weather for all successfully geocoded areas

Expected if all 46 locations are valid:

**46 × 7 = 322 rows**


In [ ]:
all_forecasts = []

valid_locations = df_locations.dropna(
    subset=["latitude", "longitude"]
)

for _, location in valid_locations.iterrows():
    forecast = get_weather(location)
    all_forecasts.extend(forecast)

df_weather = pd.DataFrame(all_forecasts)

df_weather["date"] = pd.to_datetime(
    df_weather["date"]
)

df_weather["weather"] = (
    df_weather["weather_code"]
    .map(weather_descriptions)
    .fillna("Unknown")
)

df_weather = (
    df_weather
    .sort_values(["country", "area", "date"])
    .reset_index(drop=True)
)

print("Areas processed:", df_weather["area"].nunique())
print("Forecast rows:", len(df_weather))
print("Expected if 46 areas succeeded:", 46 * 7)

df_weather.head()


## 11. Basic quality checks

Before visualization, verify:
- how many Belgian areas were processed;
- how many Moldovan areas were processed;
- whether important fields contain missing values.


In [ ]:
print(
    "Belgium areas:",
    df_weather.loc[
        df_weather["country"] == "Belgium",
        "area"
    ].nunique()
)

print(
    "Moldova areas:",
    df_weather.loc[
        df_weather["country"] == "Moldova",
        "area"
    ].nunique()
)

print()
print("Missing values:")
print(
    df_weather[
        [
            "area",
            "representative_city",
            "date",
            "temp_min",
            "temp_max",
            "precipitation_mm",
            "weather",
        ]
    ].isna().sum()
)


## 12. Function: plot one regional centre

The title shows both:
- the administrative area;
- the representative weather city.

Example:

`7-day forecast — Ungheni Raion (Ungheni)`


In [ ]:
def plot_weather(area_name, data):
    data = data.sort_values("date")

    city = data["representative_city"].iloc[0]

    fig, ax1 = plt.subplots(figsize=(12, 6))

    ax1.plot(
        data["date"],
        data["temp_min"],
        marker="o",
        label="Minimum temperature",
    )

    ax1.plot(
        data["date"],
        data["temp_max"],
        marker="o",
        label="Maximum temperature",
    )

    ax1.set_xlabel("Date")
    ax1.set_ylabel("Temperature (°C)")
    ax1.set_title(
        f"7-day forecast — {area_name} ({city})"
    )

    for date, temp, weather in zip(
        data["date"],
        data["temp_max"],
        data["weather"],
    ):
        ax1.annotate(
            weather,
            (date, temp),
            xytext=(0, 10),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )

    ax2 = ax1.twinx()

    ax2.bar(
        data["date"],
        data["precipitation_mm"],
        alpha=0.3,
        label="Precipitation",
    )

    ax2.set_ylabel("Precipitation (mm)")

    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")

    ax1.tick_params(
        axis="x",
        rotation=45,
    )

    plt.tight_layout()
    plt.show()


## 13. Default Sinapsa view: Brussels ↔ Chișinău

This is the default public pair we want to keep.

For now the notebook displays two charts.
Later the website can present them side by side.


In [ ]:
default_areas = [
    "Brussels-Capital Region",
    "Chișinău Municipality",
]

for area_name in default_areas:
    data = df_weather[
        df_weather["area"] == area_name
    ]

    plot_weather(
        area_name,
        data,
    )


## 14. Test another regional pair

Example:

**Liège Province ↔ Ungheni Raion**

Change the two names below to test other combinations.


In [ ]:
selected_belgium_area = "Liège Province"
selected_moldova_area = "Ungheni Raion"

for area_name in [
    selected_belgium_area,
    selected_moldova_area,
]:
    data = df_weather[
        df_weather["area"] == area_name
    ]

    plot_weather(
        area_name,
        data,
    )


# New pipeline

```text
46 reference areas
      ↓
representative city
      ↓
Open-Meteo Geocoding
      ↓
latitude + longitude
      ↓
Open-Meteo Forecast
      ↓
7 days × 46 locations
      ↓
Pandas
      ↓
322 forecast rows
      ↓
Brussels ↔ Chișinău default
      ↓
optional regional comparison
```

## What we deliberately removed

We no longer need:

```text
ODWB
↓
all Belgian communes
↓
pagination
```

because Sinapsa's current scope is regional rather than commune-level.

## Next logical step

Once this notebook runs correctly from top to bottom:

1. save the 46 geocoded locations so they do not need to be geocoded every run;
2. save the forecast table to CSV;
3. later store it through SQLAlchemy;
4. connect Brussels ↔ Chișinău to the website.
